# JIT Compilation

## Setup

In [1]:
import jax
import jax.numpy as jnp
import time


def f(x):
    for _ in range(50):
        x = 0.25 * x**3 + 0.75 * x
    return jnp.sum(x)

In [2]:
x = jnp.ones((1000, 1000))

start = time.perf_counter()
for _ in range(50):
    f(x).block_until_ready()
print(f"Without (jit): {(time.perf_counter() - start) / 50 * 1000:.2f} ms per call")

Without (jit): 104.94 ms per call


## Applying `jax.jit`


In [3]:
f_jit = jax.jit(f)

In [4]:
# First call (tracing + compilation)
start = time.perf_counter()
for _ in range(50):
    f_jit(x).block_until_ready()
print(
    f"First call (trace + compile): {(time.perf_counter() - start) / 50 * 1000:.2f} ms per call"
)

First call (trace + compile): 6.22 ms per call


In [5]:
# Subsequent calls (use the cached compiled function)
start = time.perf_counter()
for _ in range(50):
    f_jit(x).block_until_ready()
print(f"Subsequent calls: {(time.perf_counter() - start) / 50 * 1000:.2f} ms per call")

Subsequent calls: 3.13 ms per call


In [6]:
@jax.jit
def f_decorated(x):
    for _ in range(50):
        x = 0.25 * x**3 + 0.75 * x
    return jnp.sum(x)


_ = f_decorated(x).block_until_ready()

## Asynchronous Dispatch and `block_until_ready`

In [7]:
@jax.jit
def heavy(a):
    return a @ a


a = jnp.ones((2000, 2000))
heavy(a).block_until_ready()

Array([[2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       ...,
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.],
       [2000., 2000., 2000., ..., 2000., 2000., 2000.]], dtype=float32)

In [8]:
start = time.perf_counter()
result = heavy(a)
print(f"Call returned after: {(time.perf_counter() - start) * 1000:.3f} ms")

Call returned after: 0.177 ms


In [9]:
start = time.perf_counter()
heavy(a).block_until_ready()
print(f"Total time (dispatch + compute): {(time.perf_counter() - start) * 1000:.3f} ms")

Total time (dispatch + compute): 23.522 ms


## The Compilation Cache

In [10]:
@jax.jit
def g(x):
    return x * 2.0


# Compile for shape (10,)
start = time.perf_counter()
_ = g(jnp.ones(10)).block_until_ready()
print(f"Compiled for shape (10,): {(time.perf_counter() - start) * 1000:.2f} ms")

# Cache hit (same shape)
start = time.perf_counter()
_ = g(jnp.ones(10)).block_until_ready()
print(f"Compiled for shape (10,): {(time.perf_counter() - start) * 1000:.2f} ms")

# Recompile (new shape)
start = time.perf_counter()
_ = g(jnp.ones(20)).block_until_ready()
print(f"Recompile for shape (20,): {(time.perf_counter() - start) * 1000:.2f} ms")

Compiled for shape (10,): 47.61 ms
Compiled for shape (10,): 0.43 ms
Recompile for shape (20,): 35.58 ms


## Static Arguments

In [11]:
from functools import partial


@partial(jax.jit, static_argnums=(1,))
def apply_n_times(x, n):
    for _ in range(n):
        x = jnp.tanh(x)
    return x

In [12]:
x = jnp.ones(4)

In [13]:
start = time.perf_counter()
_ = apply_n_times(x, 3).block_until_ready()
print(f"Compiles for n=3: {(time.perf_counter() - start) * 1000:.2f} ms")

start = time.perf_counter()
_ = apply_n_times(x, 3).block_until_ready()
print(f"Cache hit: {(time.perf_counter() - start) * 1000:.2f} ms")

start = time.perf_counter()
_ = apply_n_times(x, 5).block_until_ready()
print(f"Recompiles for n=5: {(time.perf_counter() - start) * 1000:.2f} ms")

Compiles for n=3: 20.63 ms
Cache hit: 0.43 ms
Recompiles for n=5: 20.71 ms


## Pitfall: Side Effects Inside jit

In [14]:
@jax.jit
def f_with_print(x):
    print("Python print - fires at trace time only!")
    return x + 1.0

In [15]:
print("=== Call 1 ===")
_ = f_with_print(jnp.array(1.0)).block_until_ready()

print("=== Call 2 ===")
_ = f_with_print(jnp.array(2.0)).block_until_ready()

print("=== Call 3 (new shape - recompiles) ===")
_ = f_with_print(jnp.ones(3)).block_until_ready()

=== Call 1 ===
Python print - fires at trace time only!
=== Call 2 ===
=== Call 3 (new shape - recompiles) ===
Python print - fires at trace time only!


In [ ]:
@jax.jit
def f_with_debug_print(x):
    jax.debug.print("Value of x at runtime: {}", x)
    return x + 1.0


# there was a mistake in the recording: this runs fine with @jax.jit left in place too
# (debug_print just won't fire, since make_jaxpr only traces and never dispatches)
# jax.make_jaxpr(f_with_debug_print)(jnp.ones(3))

In [17]:
_ = f_with_debug_print(jnp.ones(3)).block_until_ready()
_ = f_with_debug_print(jnp.array(2.0)).block_until_ready()

Value of x at runtime: [1. 1. 1.]
Value of x at runtime: 2.0


## Pitfall: Python Control Flow on Dynamic Values Inside jit

In [18]:
@jax.jit
def broken_function(x):
    if x > 0:
        return x * 2.0
    else:
        return x

In [19]:
try:
    broken_function(jnp.array(1.0))
except Exception as e:
    print(type(e).__name__)
    print(e)

TracerBoolConversionError
Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function broken_function at C:\Users\user\AppData\Local\Temp\ipykernel_11776\458321988.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument x.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError


In [20]:
@jax.jit
def fixed_function(x):
    return jax.lax.cond(x > 0, lambda x: x * 2.0, lambda x: x, x)

In [21]:
fixed_function(jnp.array(1.0))

Array(2., dtype=float32, weak_type=True)

In [22]:
fixed_function(jnp.array(-1.0))

Array(-1., dtype=float32, weak_type=True)

## Composing jit with grad

In [ ]:
import jax.random as jrandom

In [31]:
def loss_fn(params, x, y):
    pred = params["w"] @ x + params["b"]
    return jnp.mean((pred - y) ** 2)

In [32]:
@jax.jit
def train_step(params, x, y):
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y)
    new_params = jax.tree_util.tree_map(lambda p, g: p - 0.01 * g, params, grads)
    return new_params, loss

In [33]:
# Initialize
key = jrandom.PRNGKey(0)
params = {"w": jrandom.normal(key, (1, 4)), 'b': jnp.zeros(1)}
x = jnp.ones(4)
y = jnp.array([2.0])

In [34]:
# Run one step
params, loss = train_step(params, x, y)
print(f"Loss: {loss}")
print(f"Updated w: {params["w"]}")

Loss: 1.289803147315979
Updated w: [[ 1.5999283   2.0025508  -0.45630834 -0.10133125]]


## Inspecting What Gets Compiled

In [36]:
@jax.jit
def h(x):
    return jnp.sin(x) + jnp.cos(x)

In [37]:
x = jnp.ones(4)

In [38]:
lowered = h.lower(x)
print(lowered.as_text())

module @jit_h attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<4xf32>) -> (tensor<4xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.sine %arg0 : tensor<4xf32>
    %1 = stablehlo.cosine %arg0 : tensor<4xf32>
    %2 = stablehlo.add %0, %1 : tensor<4xf32>
    return %2 : tensor<4xf32>
  }
}



## `jit` Itself Is a Primitive

In [39]:
def outer(arg):
    @jax.jit
    def inner(x):
        return x + arg * jnp.ones(1)
    return arg + inner(arg - 2.)

In [40]:
print(jax.make_jaxpr(outer)(1.0))

{ lambda ; a:f32[]. let
    b:f32[] = sub a 2.0:f32[]
    c:f32[1] = jit[
      name=inner
      jaxpr={ lambda ; a:f32[] b:f32[]. let
          d:f32[1] = broadcast_in_dim 1.0:f32[]
          e:f32[] = convert_element_type[new_dtype=float32 weak_type=False] a
          f:f32[1] = mul e d
          g:f32[] = convert_element_type[new_dtype=float32 weak_type=False] b
          c:f32[1] = add g f
        in (c,) }
    ] a b
    h:f32[] = convert_element_type[new_dtype=float32 weak_type=False] a
    i:f32[1] = add h c
  in (i,) }


## `disable_jit` for Debugging

In [54]:
@jax.jit
def f(x):
    print("Tracing!")
    return x * 2.0

In [55]:
print("=== With jit ===")
_ = f(jnp.array(1.0))
_ = f(jnp.array(2.0))

=== With jit ===
Tracing!


In [56]:
print("=== With disable_jit ===")
with jax.disable_jit():
    _ = f(jnp.array(1.0))
    _ = f(jnp.array(2.0))

=== With disable_jit ===
Tracing!
Tracing!


## Recompilation Diagnosis

In [57]:
trace_count = [0]

@jax.jit
def counted(x, n):
    trace_count[0] += 1
    return x * n

In [59]:
x = jnp.ones(4)
_ = counted(x, 3)
_ = counted(x, 3)
_ = counted(x, 5)
_ = counted(x, 3)

In [60]:
print(f"Number of traces: {trace_count[0]}")

Number of traces: 1


In [61]:
trace_count = [0]

def make_step():
    def step(x):
        trace_count[0] += 1
        return x * 2.0
    return jax.jit(step)

In [62]:
x = jnp.ones(4)
for _ in range(3):
    step = make_step()
    _ = step(x).block_until_ready()
print(f"Number of traces: {trace_count[0]}")

Number of traces: 3
